# 6 · Enhancer redundancy — Steps 1 & 2 + classification (Spark, cluster)

Enhancer sets differ per cell line (identical genes), so the c1-nearest enhancer is matched to c2 by **genomic overlap + padding**, not by id.

- **Step 1 (Spark):** nearest **active** enhancer per gene per cell line (+ coords).
- **Step 2 (Spark):** range-join each gene's c1-nearest enhancer region (± `PADDING`) to c2's own enhancers for that gene, take the genomically-nearest match, and reuse its precomputed `avg_dist` in c2's model → `dist_L1_in_c2`. No model loading.
- **Classify (pandas, via `install_deps`):** per-chromosome tertiles; `nearest_changed` = L1 and L2 do **not** overlap within padding; flag redundancy.

Writes a finished redundancy table per directed comparison to `s3a://database/enhancer_redundancy/<comparison>` and prints a **coverage report** (fraction of c1-nearest enhancers with an overlapping c2 enhancer). Notebook 7 (local, no storage) does the DESeq2 + |log2FC| analysis. See `docs/superpowers/specs/2026-06-16-enhancer-redundancy-design.md`.

In [ ]:
%%configure -f
{"executorMemory": "12G", "executorCores": 12, "ttl": "12h", "heartbeatTimeoutInSecond": 43200, "numExecutors": 3}

In [ ]:
# Install Python deps into the live Livy kernel (driver) so pandas/numpy work for
# the classification/assembly below. (The Spark joins themselves don't need this.)
def install_deps(deps):
    from pyspark import SparkFiles
    from subprocess import call
    import sys
    for package in deps:
        call([sys.executable, '-m', 'pip', 'install', '-q', '-t', SparkFiles.getRootDirectory(), package])

install_deps(['pandas', 'numpy'])

import sys
from pyspark import SparkFiles
_root = SparkFiles.getRootDirectory()
if _root not in sys.path:
    sys.path.insert(0, _root)

In [ ]:
import numpy as np
import pandas as pd
from pyspark.sql import Window
import pyspark.sql.functions as F

ACTIVE_STATES = ['TssA', 'TssAFlnk', 'TxFlnk', 'Tx', 'TxWk',
                 'EnhG', 'EnhG1', 'EnhG2', 'Enh', 'EnhA1', 'EnhA2']
QUERY_IDS = {
    "GM12878": "a1fc46a9-93f8-424f-b41d-37bfd85d3b94",
    "H1ESC":   "f7bc6dac-6aa3-49e6-a2e5-c2ff27824c81",
    "HFFC6":   "f648f805-c3a9-4cf4-a108-94e6f5fa96c1",
}
USED_PROJECTS = ['whole_all_vs_all_gm12878_fix',
                 'whole_all_vs_all_h1esc_fix',
                 'whole_all_vs_all_hffc6_fix']
COMPARISONS = [("GM12878","H1ESC"), ("H1ESC","GM12878"),
               ("H1ESC","HFFC6"),   ("HFFC6","H1ESC"),
               ("GM12878","HFFC6"), ("HFFC6","GM12878")]

PADDING = 5000   # bp tolerance for matching enhancers across cell lines (tunable)
OUTPUT_BASE = "s3a://database/enhancer_redundancy"

In [ ]:
def read_results(cell_line, query_id):
    return (spark.read.parquet(f"s3a://database/results/{query_id}")
            .withColumn("cell_line", F.lit(cell_line)))

results = None
for cl, qid in QUERY_IDS.items():
    df = read_results(cl, qid)
    results = df if results is None else results.union(df)

results = (results
           .where("avg_dist > 0 AND var_dist > 0")
           .where(F.col('project_id').isin(USED_PROJECTS)))

chromatin_states_df = (spark.read.parquet("s3a://database/chromatin_states")
                       .where(F.col('name').isin(ACTIVE_STATES)))
results.createOrReplaceTempView("results")
chromatin_states_df.createOrReplaceTempView("chromatin_states")

In [ ]:
# active (gene & enhancer both overlap an active ChromHMM state), like notebook 1
active_pairs = spark.sql("""
SELECT r.gene_id, r.gene_chr, r.enh_id, r.enh_chr, r.enh_start, r.enh_end,
       r.avg_dist, r.cell_line
FROM results r
WHERE EXISTS (SELECT 1 FROM chromatin_states cs
              WHERE cs.cell_line = r.cell_line AND cs.chrom = r.gene_chr
                AND cs.start <= r.gene_end AND cs.end >= r.gene_start)
  AND EXISTS (SELECT 1 FROM chromatin_states cs
              WHERE cs.cell_line = r.cell_line AND cs.chrom = r.enh_chr
                AND cs.start <= r.enh_end AND cs.end >= r.enh_start)
""")

In [ ]:
# Step 1: nearest active enhancer per (cell_line, gene); collect (small) to driver
w = Window.partitionBy('cell_line', 'gene_id').orderBy(F.col('avg_dist').asc())
nearest = (active_pairs
           .withColumn('rk', F.row_number().over(w))
           .where('rk = 1')
           .drop('rk'))

nearest_pd = nearest.toPandas()
for c in ['gene_chr', 'enh_chr']:
    nearest_pd[c] = nearest_pd[c].astype(str)
print(nearest_pd.shape, nearest_pd.cell_line.value_counts().to_dict())

In [ ]:
# Step 2 (overlap + padding): match L1 to c2's nearest-overlapping enhancer and
# reuse its precomputed distance in c2's model. c2 enhancers are UNFILTERED by
# activity (the old enhancer may be inactive in c2).
all_enh = (results
           .groupBy('cell_line', 'gene_id',
                    F.col('enh_chr').alias('e_chr'),
                    F.col('enh_start').alias('e_start'),
                    F.col('enh_end').alias('e_end'))
           .agg(F.min('avg_dist').alias('e_dist')))

def step2_overlap(c1, c2):
    L1 = (nearest.where(F.col('cell_line') == c1)
          .select('gene_id',
                  F.col('enh_chr').alias('L_chr'),
                  F.col('enh_start').alias('L_start'),
                  F.col('enh_end').alias('L_end')))
    e2 = (all_enh.where(F.col('cell_line') == c2)
          .select(F.col('gene_id').alias('g2'), 'e_chr', 'e_start', 'e_end', 'e_dist'))
    j = (L1.join(e2,
                 (F.col('gene_id') == F.col('g2')) &
                 (F.col('L_chr') == F.col('e_chr')) &
                 (F.col('L_start') - PADDING <= F.col('e_end')) &
                 (F.col('L_end') + PADDING >= F.col('e_start')),
                 'left')
         .select('gene_id',
                 F.col('e_dist').alias('dist_L1_in_c2'),
                 F.abs((F.col('e_start') + F.col('e_end')) / 2
                       - (F.col('L_start') + F.col('L_end')) / 2).alias('cdiff')))
    wj = Window.partitionBy('gene_id').orderBy(F.col('cdiff').asc_nulls_last())
    return (j.withColumn('rk', F.row_number().over(wj)).where('rk = 1')
            .select('gene_id', 'dist_L1_in_c2'))

l1_by_comp = {}
for c1, c2 in COMPARISONS:
    comp = f"{c1.lower()}_vs_{c2.lower()}"
    l1_by_comp[comp] = step2_overlap(c1, c2).toPandas().set_index('gene_id')['dist_L1_in_c2']
    print(comp, "-> L1-in-c2 rows:", len(l1_by_comp[comp]))

In [ ]:
# Classification (pandas) + write finished redundancy table per comparison
def chrom_tertile_thresholds(df, dist_col, chrom_col):
    out = {}
    for chrom, grp in df.groupby(chrom_col):
        out[chrom] = (float(grp[dist_col].quantile(0.33)),
                      float(grp[dist_col].quantile(0.67)))
    return out

def proximity_categories(dist_series, chrom_series, thresholds):
    cats = []
    for val, ch in zip(dist_series, chrom_series):
        t = thresholds.get(ch)
        if t is None or pd.isna(val):
            cats.append('large')
        elif val <= t[0]:
            cats.append('small')
        elif val <= t[1]:
            cats.append('mid')
        else:
            cats.append('large')
    return cats

def build_comparison(c1, c2):
    comp = f"{c1.lower()}_vs_{c2.lower()}"
    n1 = nearest_pd[nearest_pd.cell_line == c1].set_index('gene_id')
    n2 = nearest_pd[nearest_pd.cell_line == c2].set_index('gene_id')
    th_c1 = chrom_tertile_thresholds(n1.rename(columns={'avg_dist': 'd'}), 'd', 'gene_chr')
    th_c2 = chrom_tertile_thresholds(n2.rename(columns={'avg_dist': 'd'}), 'd', 'gene_chr')

    genes = n1.index.intersection(n2.index)
    df = pd.DataFrame({'gene_id': list(genes)})
    df['gene_chr'] = n1.loc[genes, 'gene_chr'].values
    df['nearest_enh_c1'] = n1.loc[genes, 'enh_id'].values
    df['avg_dist_c1'] = n1.loc[genes, 'avg_dist'].values
    df['nearest_enh_c2'] = n2.loc[genes, 'enh_id'].values
    df['avg_dist_c2'] = n2.loc[genes, 'avg_dist'].values
    df['dist_L1_in_c2'] = l1_by_comp[comp].reindex(genes).values

    # nearest_changed: L1 and L2 do NOT overlap within padding (genuinely different element)
    a_chr = n1.loc[genes, 'enh_chr'].values; a_s = n1.loc[genes, 'enh_start'].values; a_e = n1.loc[genes, 'enh_end'].values
    b_chr = n2.loc[genes, 'enh_chr'].values; b_s = n2.loc[genes, 'enh_start'].values; b_e = n2.loc[genes, 'enh_end'].values
    overlap = (a_chr == b_chr) & (a_s - PADDING <= b_e) & (a_e + PADDING >= b_s)
    df['nearest_changed'] = ~overlap
    df['L1_found_in_c2'] = ~pd.isna(df['dist_L1_in_c2'])

    df['proximity_category_c1'] = proximity_categories(df['avg_dist_c1'], df['gene_chr'], th_c1)
    df['proximity_category_c2'] = proximity_categories(df['avg_dist_c2'], df['gene_chr'], th_c2)
    df['proximity_of_L1_in_c2'] = proximity_categories(df['dist_L1_in_c2'], df['gene_chr'], th_c2)

    old_far = df['proximity_of_L1_in_c2'].eq('large')
    new_close = df['proximity_category_c2'].eq('small')
    df['is_redundancy'] = df['nearest_changed'] & old_far & new_close
    df['is_nonredundant_switcher'] = df['nearest_changed'] & old_far & ~new_close

    df.insert(0, 'comparison', comp); df.insert(1, 'c1', c1); df.insert(2, 'c2', c2)
    return df

for c1, c2 in COMPARISONS:
    comp = f"{c1.lower()}_vs_{c2.lower()}"
    t = build_comparison(c1, c2)
    print(f"{c1}->{c2}: genes={len(t)}  redundancy={int(t.is_redundancy.sum())}  "
          f"coverage(L1 found in c2)={t['L1_found_in_c2'].mean():.3f}")
    spark.createDataFrame(t).repartition(1).write.mode('overwrite').parquet(f"{OUTPUT_BASE}/{comp}")

print("done — pull", OUTPUT_BASE, "/* to data/whole_chromosomes/enhancer_redundancy/ for notebook 7")